In [30]:
import math
import torch
import torch.nn as nn
from modules import *
from einops import einsum, rearrange

In [ ]:
class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr: float = 1e-3, betas: tuple[float] = (0.9, 0.999), eps: float = 1e-8, weight_decay: float = 0.02):
        if lr < 0.0: raise ValueError(f"Invalid learning rate: {lr}")
        if eps < 0.0: raise ValueError(f"Invalid epsilon value: {eps}")
        if not 0.0 <= betas[0] < 1.0: raise ValueError(f"Invalid beta1 parameter: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0: raise ValueError(f"Invalid beta2 parameter: {betas[1]}")
        if weight_decay < 0.0: raise ValueError(f"Invalid weight_decay value: {weight_decay}")

        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)                
        super(AdamW, self).__init__(params, defaults)

    def step(self, closure = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            beta1, beta2 = group['betas']
            alpha = group['lr']
            eps = group['eps']
            lambd = group['weight_decay']

            for p in group['params']:
                if p.grad is None: continue

            state = self.state[p]
            g = p.grad

            if len(state) == 0:
                state['step'] = 0
                state['m'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                state['v'] = torch.zeros_like(p, memory_format=torch.preserve_format)

            state['step'] += 1
            t = state['step']
            m, v = state['m'], state['v']

            # compute the moments
            m.mul_(beta1).add_(g, alpha = 1 - beta1)
            v.mul_(beta2).addcmul(g, g, value = 1 - beta2)

            # update the learning rate
            correction_numerator = 1 - beta1 ** t
            correction_denominator = 1 - beta2 ** t
            alpha_updated = alpha * (correction_numerator ** 0.5) / correction_denominator

            # update parameters and apply weight decay
            p.addcdiv_(m, v.sqrt().add_(eps), value = -alpha_updated)
            if lambd != 0: p.add_(p, alpha = -alpha_updated * lambd)

        return loss

SyntaxError: incomplete input (2862699629.py, line 14)